# 03 · Boundary Detection
Cosine dissimilarity boundary detector and Global Load Balancing auxiliary loss.

**Papers:** DLCM ([2512.24617](https://arxiv.org/abs/2512.24617)) Eq. 5-10, H-Net ([2507.07955](https://arxiv.org/abs/2507.07955))

In [ ]:
import torch, sys
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
sys.path.insert(0, '..')
from src.model import BoundaryDetector, GlobalLoadBalancer, ConceptLMConfig

## 1. Cosine dissimilarity boundary scoring (DLCM Eq. 5-6)

In [ ]:
cfg = ConceptLMConfig(d_token=256, d_scan=128)
det = BoundaryDetector(cfg)

torch.manual_seed(42)
L = 24
v1 = F.normalize(torch.randn(256), dim=0)
v2 = F.normalize(torch.randn(256), dim=0)
v2 = F.normalize(v2 - (v2 @ v1) * v1, dim=0)  # orthogonalize

H = torch.zeros(1, L, 256)
H[0, :12]  = v1 + 0.15 * torch.randn(12, 256)
H[0, 12:]  = v2 + 0.15 * torch.randn(12, 256)

b, p = det(H, training=False)
print("Boundary probabilities:")
for i, pi in enumerate(p[0].tolist()):
    flag = " <<< TRUE BOUNDARY" if i == 12 else ""
    print(f"  pos {i:2d}: p={pi:.4f}{flag}")

colors = ['#C00000' if pi > 0.5 else '#2E75B6' for pi in p[0].tolist()]
plt.figure(figsize=(12, 3))
plt.bar(range(L), p[0].numpy(), color=colors)
plt.axhline(0.5, color='black', linestyle='--', label='threshold')
plt.axvline(11.5, color='green', linestyle=':', lw=2, label='true boundary')
plt.xlabel('Token position'); plt.ylabel('Boundary probability'); plt.legend()
plt.title('Cosine Dissimilarity Boundary Detection (DLCM Eq. 6)')
plt.tight_layout(); plt.show()

## 2. Training (Bernoulli) vs. inference (hard threshold)

In [ ]:
def sharpen(p, tau=0.1):
    return torch.sigmoid((p - 0.5) / tau)

p_sharp = sharpen(p[0], tau=cfg.train_temperature)
b_train = torch.bernoulli(p_sharp)
b_infer = (p[0] >= 0.5).float()

print("Training  boundaries:", b_train.nonzero().squeeze().tolist())
print("Inference boundaries:", b_infer.nonzero().squeeze().tolist())
print("Ground truth:        [0, 12]  (pos 0 always = 1 by design)")

## 3. Global Load Balancing loss (DLCM Eq. 8-10)

In [ ]:
balancer = GlobalLoadBalancer(target_ratio=4)
print(f"Aux loss vs. boundary count  (L={L}, target R=4 => ~6 boundaries):")
for n_b in [1, 3, 6, 12, 20]:
    b_sim = torch.zeros(1, L); b_sim[0, 0] = 1
    if n_b > 1:
        idx = torch.randperm(L-1)[:n_b-1] + 1
        b_sim[0, idx] = 1
    p_sim = b_sim.clone().clamp(0.01, 0.99)
    loss  = balancer(b_sim, p_sim)
    ratio = L / max(1, b_sim.sum().item())
    print(f"  n_boundaries={n_b:2d}  actual_ratio={ratio:.1f}  aux_loss={loss.item():.5f}")

## 4. Learned vs. rule-based boundaries

In [ ]:
import re
text = ("The encoder produces representations H. "
        "These feed into the boundary detector. "
        "It scores each position using cosine similarity. "
        "High-dissimilarity positions become concept boundaries.")

rule_segs = re.split(r'(?<=[.!?])\s+', text)
print("Rule-based segments (punctuation-based):")
for i, s in enumerate(rule_segs):
    print(f"  [{i}] {s}")
print()
print("Learned (DLCM) advantages:")
print("  - No punctuation rules or language-specific NLP pipeline")
print("  - Adapts granularity to information density per content type")
print("  - Operates in latent space, not character space")
print("  - Jointly optimized with the language model objective")